# Calendar Dimension (adb_caresync.silver.calendar)

Builds a standard date dimension with one row per day from 1940-01-01 to 2030-12-31, including keys, calendar, ISO week, and flag attributes. Each run fully overwrites the table.

In [0]:
# Parameters
TARGET_TABLE = "adb_caresync.silver.calendar"
START_DATE = "1940-01-01"
END_DATE = "2030-12-31"

calendar_sql = f"""
WITH dates AS (
  SELECT explode(sequence(DATE'{START_DATE}', DATE'{END_DATE}', INTERVAL 1 DAY)) AS date
),
base AS (
  SELECT
    date,
    ((dayofweek(date) + 5) % 7) + 1 AS iso_dow   -- 1 = Monday ... 7 = Sunday
  FROM dates
)
SELECT
  CAST(date_format(date, 'yyyyMMdd') AS INT)              AS date_key,
  date,
  -- Year
  year(date)                                              AS year,
  make_date(year(date), 1, 1)                             AS year_start_date,
  make_date(year(date), 12, 31)                           AS year_end_date,
  CASE WHEN (year(date) % 4 = 0 AND year(date) % 100 <> 0) OR year(date) % 400 = 0
       THEN true ELSE false END                           AS is_leap_year,
  -- Half year
  CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END             AS half_year,
  concat('H', CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END) AS half_year_name,
  -- Quarter
  quarter(date)                                           AS quarter,
  concat('Q', quarter(date))                              AS quarter_name,
  concat(year(date), '-Q', quarter(date))                 AS year_quarter,
  CAST(date_trunc('QUARTER', date) AS DATE)               AS quarter_start_date,
  date_sub(add_months(CAST(date_trunc('QUARTER', date) AS DATE), 3), 1) AS quarter_end_date,
  datediff(date, CAST(date_trunc('QUARTER', date) AS DATE)) + 1 AS day_of_quarter,
  -- Month
  month(date)                                             AS month,
  date_format(date, 'MMMM')                               AS month_name,
  date_format(date, 'MMM')                                AS month_short_name,
  CAST(date_format(date, 'yyyyMM') AS INT)                AS year_month_key,
  date_format(date, 'yyyy-MM')                            AS year_month,
  date_format(date, 'MMM yyyy')                           AS month_year_name,
  CAST(date_trunc('MONTH', date) AS DATE)                 AS month_start_date,
  last_day(date)                                          AS month_end_date,
  day(last_day(date))                                     AS days_in_month,
  -- Week (ISO 8601: weeks start Monday, week 1 contains the first Thursday)
  weekofyear(date)                                        AS week_of_year,
  year(date_add(date, 4 - iso_dow))                       AS iso_year,
  concat(year(date_add(date, 4 - iso_dow)), '-W', lpad(weekofyear(date), 2, '0')) AS iso_year_week,
  date_sub(date, iso_dow - 1)                             AS week_start_date,
  date_add(date, 7 - iso_dow)                             AS week_end_date,
  CAST(ceil((day(date) + ((dayofweek(CAST(date_trunc('MONTH', date) AS DATE)) + 5) % 7)) / 7.0) AS INT) AS week_of_month,
  -- Day
  day(date)                                               AS day_of_month,
  dayofyear(date)                                         AS day_of_year,
  iso_dow                                                 AS day_of_week,
  date_format(date, 'EEEE')                               AS day_name,
  date_format(date, 'EEE')                                AS day_short_name,
  -- Flags
  iso_dow IN (6, 7)                                       AS is_weekend,
  iso_dow NOT IN (6, 7)                                   AS is_weekday,
  date = CAST(date_trunc('MONTH', date) AS DATE)          AS is_month_start,
  date = last_day(date)                                   AS is_month_end,
  date = CAST(date_trunc('QUARTER', date) AS DATE)        AS is_quarter_start,
  date = date_sub(add_months(CAST(date_trunc('QUARTER', date) AS DATE), 3), 1) AS is_quarter_end,
  (month(date) = 1 AND day(date) = 1)                     AS is_year_start,
  (month(date) = 12 AND day(date) = 31)                   AS is_year_end
FROM base
"""

calendar_df = spark.sql(calendar_sql)

(calendar_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE))

spark.sql(f"COMMENT ON TABLE {TARGET_TABLE} IS 'Standard calendar dimension, one row per day from {START_DATE} to {END_DATE}. date_key = yyyyMMdd. Weeks follow ISO 8601 (Monday start, day_of_week 1=Mon..7=Sun).'")

display(spark.table(TARGET_TABLE).orderBy("date").limit(10))

In [0]:
from pyspark.sql import functions as F

cal = spark.table(TARGET_TABLE)

# Expected day count computed independently of the table
expected_days = spark.sql(f"SELECT datediff(DATE'{END_DATE}', DATE'{START_DATE}') + 1 AS n").first()["n"]

summary = cal.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("date_key").alias("distinct_date_keys"),
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
    F.sum(F.col("is_leap_year").cast("int")).alias("leap_year_days"),
    F.sum(F.col("is_weekend").cast("int")).alias("weekend_days"),
).withColumn("expected_days", F.lit(expected_days))

display(summary)

# Spot-check known dates (2024-12-30 is Monday of ISO week 2025-W01; 2021-01-03 is Sunday of 2020-W53)
display(cal.filter(F.col("date").isin("1940-01-01", "2021-01-03", "2024-02-29", "2024-12-30", "2030-12-31"))
          .select("date_key", "date", "day_name", "day_of_week", "week_of_year", "iso_year_week",
                  "week_start_date", "week_of_month", "quarter_name", "is_leap_year", "is_weekend")
          .orderBy("date"))